# MedBook: train the joint XLM-R model on Colab

1. Runtime > Change runtime type > **T4 GPU**.
2. Upload the whole `medbook` folder to Google Drive under `MyDrive/medbook` (or change `ROOT` below).
3. Run all cells. Training takes about 10 minutes on a T4.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/medbook'
!ls $ROOT

In [ ]:
!pip -q install transformers sentencepiece scikit-learn sklearn-crfsuite
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Baseline (for comparison)

In [ ]:
%cd $ROOT/models
!python baseline.py --data ../data/dataset_v1 --out ../results/baseline_v1

## Train XLM-R (selects the best epoch on val, reports test and free-form)

In [ ]:
%cd $ROOT/models
!python train_xlmr.py --data ../data/dataset_v1 --out ../results/xlmr_v1 --epochs 5 --batch_size 32

## Look at the free-form mistakes

In [ ]:
import json, pandas as pd
err = [json.loads(l) for l in open(f'{ROOT}/results/xlmr_v1/errors_freeform.jsonl', encoding='utf-8')]
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(err)[['text','intent','pred_intent','department','pred_department']]

## Try your own messages

In [ ]:
import sys; sys.path.insert(0, f'{ROOT}/models')
from predict_xlmr import Parser
p = Parser(f'{ROOT}/results/xlmr_v1/model')
for t in ['assalam o alaikum, mere bete ko 2 din se bukhar hai, kal subah ka time mil sakta hai?',
          'مجھے کل ڈاکٹر عائشہ سے appointment چاہیے',
          'mera appointment MB-1042 cancel kar dein']:
    print(p(t))

## Optional: final model trained on train + val
Use the epoch count that was best on val above (see `train_log.json`).

In [ ]:
# !python train_xlmr.py --data ../data/dataset_v1 --out ../results/xlmr_v1_final --epochs 4 --train_on_val